# Feature Engineering Challenge

**Feature engineering (FE)** is the craft of turning raw columns into inputs that expose a problem's structure to a model. A learner can only recombine the signals you hand it; if the true signal lives in a *ratio*, a *calendar effect*, or a *per-group statistic*, then giving the model those quantities directly is often worth far more than tuning hyper-parameters.

> *"Applied machine learning is basically feature engineering."* — attributed to Andrew Ng

In this notebook we:

1. **Synthesize** a raw e-commerce order table whose `profit` is secretly driven by engineered quantities.
2. Fit a **baseline** RandomForest on the raw *numeric* columns only and record its cross-validated $R^2$.
3. Add three families of features one at a time — **(a) ratios/interactions**, **(b) date parts**, **(c) group aggregations** — measuring the *incremental* lift after each.
4. Compare everything in a table + bar chart and reason about **which features helped most and why**.

**Why FE often beats model tuning.** Swapping models or grid-searching hyper-parameters usually moves a decent pipeline by a *fraction* of a percent, because every model is fitting the *same* information. A new feature can add information that was previously not expressible at all — shifting the score by a large margin.

In [ ]:
import numpy as np                                   # numeric arrays + the RNG that drives all synthetic data
import pandas as pd                                  # the DataFrame we engineer features on
import matplotlib.pyplot as plt                      # plotting
import seaborn as sns                                # nicer statistical plots / styling
from sklearn.ensemble import RandomForestRegressor   # flexible model: captures nonlinearities out of the box
from sklearn.model_selection import KFold, cross_val_score  # honest, averaged evaluation

# --- Reproducibility: seed every source of randomness we touch ---
SEED = 42
np.random.seed(SEED)               # legacy global RNG (belt-and-suspenders)
rng = np.random.default_rng(SEED)  # modern Generator: this object generates ALL synthetic data below

# Cosmetic only (does not affect results): consistent, readable plots.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Synthesize a raw dataset

We build an e-commerce order log. Each row is one order with:

- a **datetime** — `order_date`
- **numeric** columns — `unit_price`, `base_cost`, `competitor_price`, `quantity`, `marketing_spend`, `shipping_cost`
- a **categorical / group** column — `category` (Electronics, Clothing, ...)
- the **target** — `profit` (a regression target)

The trick: we define the *true* profit as a function of quantities that are **not** raw columns — a price-to-cost **ratio**, **weekend / holiday** calendar effects, and a **per-category** premium:

$$\text{profit} \approx 10\cdot\frac{\text{price}}{\text{cost}} \;-\; 8\cdot\frac{\text{price}}{\text{competitor}} \;+\; 5\cdot\text{weekend} \;+\; 8\cdot\text{holiday} \;+\; 0.3\cdot\text{premium}_{\text{cat}}\cdot\text{qty} \;+\; \varepsilon$$

Because the signal literally lives in engineered space, recreating these quantities should measurably help the model — that is the whole point of the exercise.

In [ ]:
# ==========================================================================
# Synthesize a raw e-commerce order table.
# The TRUE target (profit) is built from ENGINEERED quantities on purpose,
# so that recreating those quantities later will visibly improve the model.
# ==========================================================================
N = 3000  # number of orders (rows)

# --- Categorical / group column -------------------------------------------
categories = ['Electronics', 'Clothing', 'Home', 'Books', 'Toys']
# Each category has a latent typical PRICE level and a latent PROFIT premium.
# NOTE: premium is deliberately NOT aligned with price level (cheap Books are
# high-premium; pricey Electronics are low-premium). So price MAGNITUDE does not
# reveal the premium -- only knowing the CATEGORY does. A per-category
# aggregation recovers exactly that identity; the raw numeric columns cannot.
cat_price_level = {'Electronics': 120, 'Clothing': 45, 'Home': 70, 'Books': 20, 'Toys': 35}
cat_premium     = {'Electronics': 4,   'Clothing': 22, 'Home': 9,  'Books': 28, 'Toys': 14}
category = rng.choice(categories, size=N, p=[0.25, 0.25, 0.20, 0.15, 0.15])  # (N,) object array

# --- Numeric columns --------------------------------------------------------
price_level      = np.array([cat_price_level[c] for c in category])  # (N,) latent price anchor per row
# WIDE multiplier -> per-row prices overlap heavily across categories, so a raw
# unit_price is a POOR category indicator. The per-category MEAN, however, stays
# crisp -- that denoised group signal is what the aggregation step will add.
unit_price       = price_level * rng.uniform(0.45, 1.55, N)         # noisy (overlapping) around the anchor
base_cost        = unit_price  * rng.uniform(0.35, 0.75, N)          # cost is some fraction of the price
competitor_price = unit_price  * rng.uniform(0.85, 1.20, N)          # rival's price, near ours
quantity         = rng.integers(1, 8, N).astype(float)              # units per order, 1..7
marketing_spend  = rng.uniform(0, 50, N)                            # ad money attributed to the order
shipping_cost    = rng.uniform(2, 15, N)                            # logistics cost (a pure-noise column)

# --- Datetime column --------------------------------------------------------
# Spread orders across two full years so month / weekday patterns are learnable.
start = pd.Timestamp('2023-01-01')
order_date = start + pd.to_timedelta(rng.integers(0, 730, N), unit='D')  # DatetimeIndex, (N,)

# --- Hidden drivers of profit (these ARE the engineered quantities) ---------
margin_ratio = unit_price / base_cost                                 # >1: markup over cost
price_ratio  = unit_price / competitor_price                          # <1 => we undercut the rival
is_weekend   = np.asarray(order_date.dayofweek >= 5, dtype=float)     # Sat/Sun shopping bump
holiday      = np.isin(order_date.month, [11, 12]).astype(float)      # Nov/Dec holiday-season bump
premium      = np.array([cat_premium[c] for c in category])          # latent per-category profit premium

# --- Assemble the target from those drivers + small irreducible noise -------
noise = rng.normal(0, 3, N)
profit = (
    10.0 * margin_ratio          # ratio driver: fatter markup -> more profit
  - 8.0  * price_ratio           # being pricier than the rival hurts profit
  + 5.0  * is_weekend            # weekend orders are more profitable
  + 8.0  * holiday               # holiday season lifts profit
  + 0.30 * premium * quantity    # category premium, AMPLIFIED by how many units sold
  + 0.4  * quantity              # a little raw-quantity signal
  + 0.02 * marketing_spend       # marketing has a mild effect
  + noise                        # irreducible randomness
)

# --- Raw dataframe: ONLY the columns a data engineer would hand you ---------
# We deliberately DO NOT include margin_ratio / is_weekend / premium / etc.
# Those must be engineered from the raw columns in the cells that follow.
df = pd.DataFrame({
    'order_date': order_date,
    'category': category,
    'unit_price': unit_price,
    'base_cost': base_cost,
    'competitor_price': competitor_price,
    'quantity': quantity,
    'marketing_spend': marketing_spend,
    'shipping_cost': shipping_cost,
    'profit': profit,            # <-- regression target
})

print(f'raw dataframe: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# A quick look at the target and at one relationship the baseline CANNOT see.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Left: distribution of the target 'profit'.
sns.histplot(df['profit'], bins=40, ax=ax[0], color='#4c72b0')
ax[0].set_title('Target distribution: profit')

# Right: mean profit varies a LOT by category -- but 'category' is TEXT, so a
# numeric-only model is blind to it until we aggregate it into numbers.
means = df.groupby('category')['profit'].mean().sort_values()   # mean profit per category
ax[1].bar(means.index, means.values, color='#dd8452')
ax[1].set_title('Mean profit by category (invisible to a numeric-only model)')
ax[1].set_ylabel('mean profit')
ax[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 2. Baseline — raw numeric features only

Before engineering anything we fix an **evaluation protocol** and get an honest starting score. We use 5-fold cross-validation (shuffled, seeded) and report the mean $R^2$ across folds.

Cross-validation matters here: we will keep adding features, and a single train/test split is noisy enough that small gains could be luck. Averaging over folds makes the comparison trustworthy. Recall $R^2 = 1$ is perfect and $R^2 = 0$ is no better than always predicting the mean.

The baseline sees only the numeric columns. It has **no access** to the datetime or the category text — so any signal hiding in the calendar or in category identity is, right now, completely invisible to it.

In [ ]:
# --------------------------------------------------------------------------
# Evaluation protocol (identical for EVERY experiment so scores are comparable):
#   * 5-fold cross-validation, shuffled, seeded (same folds every call).
#   * RandomForestRegressor (captures nonlinearities / interactions natively).
#   * Metric: R^2 averaged across the 5 folds.
# --------------------------------------------------------------------------
y = df['profit']                                          # regression target, shape (N,)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)   # fixed fold assignment across experiments

def cv_r2(feature_cols):
    """Cross-validated mean & std R^2 of a fresh RandomForest trained on feature_cols."""
    X = df[feature_cols]                                  # (N, len(feature_cols)) feature matrix
    model = RandomForestRegressor(
        n_estimators=150, random_state=SEED, n_jobs=-1    # 150 trees; seeded; use all cores
    )
    scores = cross_val_score(model, X, y, cv=cv, scoring='r2')  # one R^2 per fold -> shape (5,)
    return scores.mean(), scores.std()                    # summarize across folds

# --- BASELINE: raw NUMERIC columns only ------------------------------------
# order_date (datetime) and category (text) are non-numeric, so a plain model
# cannot consume them yet. This is the honest starting point most people begin at.
baseline_features = ['unit_price', 'base_cost', 'competitor_price',
                     'quantity', 'marketing_spend', 'shipping_cost']

base_mean, base_std = cv_r2(baseline_features)
print(f'Baseline (raw numeric only): R^2 = {base_mean:.3f} +/- {base_std:.3f}')

# Track every experiment so we can tabulate / plot the progression at the end.
results = [('baseline (raw)', base_mean, base_std)]

## 3a. Ratios & interactions

Tree ensembles split on one feature at a time. A relationship like $\frac{\text{price}}{\text{cost}}$ therefore takes many nested, axis-aligned splits to approximate — the model has to *reconstruct* a division from staircase cuts. Precomputing the ratio hands it that structure for free.

We add:

- `margin_ratio = unit_price / base_cost` — markup over cost (a true driver)
- `price_ratio  = unit_price / competitor_price` — how we compare to the rival (a true driver)
- `profit_per_unit = unit_price - base_cost` — absolute per-unit margin
- `revenue = unit_price * quantity` — a price x quantity interaction

**Leakage check:** each value is computed *within a single row* from that row's own columns. No information flows between rows or from the target, so these are safe.

In [ ]:
# --------------------------------------------------------------------------
# (a) RATIOS & INTERACTIONS between numeric columns.
# Leakage-safe: every feature is a function of a SINGLE ROW's own values only.
# --------------------------------------------------------------------------
df['margin_ratio']    = df['unit_price'] / df['base_cost']         # markup over cost (matches a true driver)
df['price_ratio']     = df['unit_price'] / df['competitor_price']  # our price vs the rival's
df['profit_per_unit'] = df['unit_price'] - df['base_cost']         # absolute margin per unit
df['revenue']         = df['unit_price'] * df['quantity']          # price x quantity interaction

# Accumulate: baseline features PLUS the new ratio/interaction features.
ratio_features = baseline_features + ['margin_ratio', 'price_ratio',
                                      'profit_per_unit', 'revenue']

r_mean, r_std = cv_r2(ratio_features)
print(f'+ ratios / interactions:     R^2 = {r_mean:.3f} +/- {r_std:.3f}  (delta {r_mean - base_mean:+.3f})')
results.append(('+ ratios', r_mean, r_std))

## 3b. Date parts

A raw timestamp is almost useless to a tree — the number of seconds since 1970 is just a big monotonic value. Its **calendar components**, however, carry seasonal signal. Our target has a weekend bump and a Nov/Dec holiday bump, so extracting those parts should pay off.

We extract `year`, `month`, `day`, `weekday`, and `is_weekend` from `order_date` via the pandas `.dt` accessor.

**Leakage check:** every part depends only on that row's own date — safe.

In [ ]:
# --------------------------------------------------------------------------
# (b) DATE PARTS extracted from the datetime column.
# Leakage-safe: each part is derived from that row's own date only.
# --------------------------------------------------------------------------
d = df['order_date'].dt                          # datetime accessor -> exposes calendar components
df['year']       = d.year                        # long-term level / trend
df['month']      = d.month                       # 1..12 -> captures the Nov/Dec holiday lift
df['day']        = d.day                         # 1..31 -> day of month
df['weekday']    = d.dayofweek                   # 0=Mon .. 6=Sun
df['is_weekend'] = (d.dayofweek >= 5).astype(int)  # 1 on Sat/Sun -> matches a true driver

# Accumulate: previous features PLUS the calendar parts.
dateparts_features = ratio_features + ['year', 'month', 'day', 'weekday', 'is_weekend']

dp_mean, dp_std = cv_r2(dateparts_features)
print(f'+ date parts:                R^2 = {dp_mean:.3f} +/- {dp_std:.3f}  (delta {dp_mean - r_mean:+.3f})')
results.append(('+ date parts', dp_mean, dp_std))

## 3c. Group aggregations

The `category` column is text, so the model cannot use it directly. We summarize each category into numbers with `groupby('category').transform(...)`, which computes a per-group statistic and broadcasts it back onto every row:

- `cat_mean_price` — the category's typical unit price (encodes category identity)
- `cat_mean_quantity` — its typical basket size
- `cat_order_count` — how common the category is

This injects the hidden **per-category premium** that drives a large chunk of profit.

**Leakage check — important:** we aggregate *feature* columns (`unit_price`, `quantity`), **not the target**. Aggregating the *target* by group (mean / target encoding) leaks test information into training and must be done fold-by-fold inside a pipeline. Feature-only aggregation is safe; for full rigor it would still be recomputed inside each CV fold rather than once on all rows.

In [ ]:
# --------------------------------------------------------------------------
# (c) GROUP AGGREGATIONS: per-category statistics via groupby(...).transform.
# transform broadcasts each group's statistic back to every row in that group,
# so the output aligns 1:1 with df and stays the same length.
#
# LEAKAGE-SAFETY: we aggregate FEATURE columns (price, quantity), NOT the target.
# Aggregating the TARGET by group (target/mean encoding) WOULD leak and must be
# done fold-wise. We avoid that here on purpose.
# --------------------------------------------------------------------------
g = df.groupby('category')
df['cat_mean_price']    = g['unit_price'].transform('mean')   # typical price -> encodes category identity
df['cat_mean_quantity'] = g['quantity'].transform('mean')     # typical basket size for the category
df['cat_order_count']   = g['unit_price'].transform('count')  # how popular the category is (group size)

# Accumulate: everything so far PLUS the per-category aggregations.
agg_features = dateparts_features + ['cat_mean_price', 'cat_mean_quantity', 'cat_order_count']

a_mean, a_std = cv_r2(agg_features)
print(f'+ group aggregations:        R^2 = {a_mean:.3f} +/- {a_std:.3f}  (delta {a_mean - dp_mean:+.3f})')
results.append(('+ aggregations', a_mean, a_std))

## 4. Results — incremental lift

Now we line up the four cross-validated scores and inspect where the gains came from: baseline -> +ratios -> +date parts -> +aggregations.

In [ ]:
# Collect every experiment into a tidy comparison table.
res_df = pd.DataFrame(results, columns=['stage', 'cv_r2', 'std'])
res_df['gain_vs_prev'] = res_df['cv_r2'].diff().fillna(0.0)          # step-by-step improvement
res_df['gain_vs_base'] = res_df['cv_r2'] - res_df['cv_r2'].iloc[0]   # cumulative vs the baseline
print(res_df.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

# Bar chart of the CV R^2 progression, with std error bars and value labels.
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(res_df['stage'], res_df['cv_r2'],
              yerr=res_df['std'], capsize=4,
              color=['#b0b0b0', '#6baed6', '#3182bd', '#08519c'])
ax.set_ylabel('cross-validated R^2')
ax.set_title('Feature engineering lifts model performance, step by step')
ax.set_ylim(0, 1)
for bar, val in zip(bars, res_df['cv_r2']):            # annotate each bar with its score
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02, f'{val:.3f}', ha='center')
plt.tight_layout()
plt.show()

# Headline number: baseline -> fully engineered.
first = res_df['cv_r2'].iloc[0]
last  = res_df['cv_r2'].iloc[-1]
print(f'\nTotal improvement from feature engineering: '
      f'{first:.3f} -> {last:.3f}  (+{last - first:.3f} R^2)')

## 5. Takeaways

- **Feature engineering added information the model could not otherwise reach.** The datetime and category columns were dead weight until we transformed them into calendar parts and per-group statistics.
- **Group aggregations give the biggest jump**, because the per-category *premium* was completely hidden from the numeric baseline and the wide price overlap made category identity impossible to infer from raw price alone; the group mean supplies that identity cleanly. **Date parts** add a solid, moderate lift (the weekend / holiday signal was otherwise absent). **Ratios** give the smallest lift, since a tree can already *partly* reconstruct `price / cost` from the raw columns.
- **Leakage discipline is part of FE.** Row-wise ratios and calendar parts are safe by construction; feature aggregations are safe; but target-based encodings must be computed inside each fold.
- **Why FE often beats tuning:** hyper-parameter search re-fits the *same* information and usually moves the score by fractions of a percent. A good feature adds *new* information and can move it by a lot — exactly what we saw going from the baseline to the fully engineered feature set.